# Severity lab — tuning and validating the OpenCV thresholds

The severity index is the project's quantitative claim, so its thresholds should be
*measured into place*, not guessed. This notebook is the workbench for that:

1. Look at the HSV distribution of healthy, chlorotic and necrotic tissue.
2. Sweep each threshold and watch the error against known ground truth.
3. Stress the pipeline with the things that break colour thresholding in the field —
   colour casts, underexposure, cluttered backgrounds.

Everything here uses `app.vision.synthetic.make_leaf`, whose damaged fraction is known by
construction. When you have real annotated leaves, drop them into `data/samples/annotated/`
with a `truth.csv` of `filename,damaged_pct` and point the last section at it — the rest of
the notebook works unchanged.

In [ ]:
from _bootstrap import setup, show_image

ROOT = setup()

import cv2
import matplotlib.pyplot as plt
import numpy as np

from app.core.logging import configure_logging
from app.vision.severity import (
    DEFAULT_CONFIG,
    SeverityConfig,
    analyse,
    classify_tier,
    render_overlay,
)
from app.vision.synthetic import make_leaf

configure_logging("ERROR")
print(DEFAULT_CONFIG)

## 1. What the tissue classes actually look like in HSV

Hue in OpenCV is 0-179, not 0-359 — the single most common source of off-by-two-times bugs
in colour masking code. The bands below are the ones encoded in `SeverityConfig`.

In [ ]:
leaf = make_leaf(damaged_fraction=0.3, lesion_count=18, chlorosis_share=0.35, seed=101)
working = cv2.bilateralFilter(leaf.image, 7, 45, 45)
hsv = cv2.cvtColor(working, cv2.COLOR_BGR2HSV)

inside = leaf.leaf_mask > 0
necrotic_truth = (leaf.lesion_mask > 0) & inside

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, channel, name, limit in zip(axes, range(3), ("hue", "saturation", "value"), (180, 256, 256), strict=True):
    ax.hist(hsv[:, :, channel][inside & ~necrotic_truth], bins=60, range=(0, limit),
            alpha=0.65, label="healthy + halo", color="#2e7d32", density=True)
    ax.hist(hsv[:, :, channel][necrotic_truth], bins=60, range=(0, limit),
            alpha=0.65, label="damaged", color="#b71c1c", density=True)
    ax.set_title(f"{name} inside the leaf")
    ax.legend(fontsize=8)

for bound in (DEFAULT_CONFIG.green_hue_lo, DEFAULT_CONFIG.green_hue_hi,
              DEFAULT_CONFIG.yellow_hue_lo, DEFAULT_CONFIG.yellow_hue_hi,
              DEFAULT_CONFIG.brown_hue_hi):
    axes[0].axvline(bound, color="black", linestyle="--", linewidth=0.8)
figure.tight_layout();

## 2. Threshold sensitivity

For each candidate value, measure the mean absolute error against ground truth across a
range of damage levels. A flat curve means the threshold is not load-bearing; a sharp one
means it is, and deserves a comment in the code explaining the choice.

In [ ]:
from dataclasses import replace

FRACTIONS = np.linspace(0.02, 0.6, 10)
LEAVES = [make_leaf(damaged_fraction=float(f), lesion_count=max(int(f * 60), 2), seed=55)
          for f in FRACTIONS]


def mean_abs_error(config: SeverityConfig) -> float:
    errors = []
    for leaf in LEAVES:
        report, _ = analyse(leaf.image, config)
        errors.append(abs(report.severity_pct - leaf.true_damaged_fraction * 100))
    return float(np.mean(errors))


SWEEPS = {
    "green_hue_lo": range(20, 45, 4),
    "green_hue_hi": range(80, 105, 4),
    "green_min_saturation": range(25, 80, 8),
    "leaf_min_saturation": range(12, 50, 6),
    "dark_value_max": range(35, 90, 8),
    "yellow_min_saturation": range(40, 100, 10),
}

baseline = mean_abs_error(DEFAULT_CONFIG)
print(f"baseline mean abs error: {baseline:.2f} pp\n")

figure, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (field, values) in zip(axes.ravel(), SWEEPS.items(), strict=True):
    xs = list(values)
    ys = [mean_abs_error(replace(DEFAULT_CONFIG, **{field: value})) for value in xs]
    ax.plot(xs, ys, marker="o", color="#1565c0")
    ax.axhline(baseline, color="grey", linestyle="--", linewidth=0.8)
    ax.axvline(getattr(DEFAULT_CONFIG, field), color="#2e7d32", linewidth=1.2)
    ax.set_title(field, fontsize=9)
    ax.set_ylabel("MAE (pp)", fontsize=8)
    best = xs[int(np.argmin(ys))]
    print(f"{field:<24} current {getattr(DEFAULT_CONFIG, field):>4}  best {best:>4}  "
          f"MAE {min(ys):.2f} pp")
figure.tight_layout();

## 3. Tier confusion — the metric that actually matters

Being 2 pp out on the severity number is harmless. Landing in the wrong *tier* is not,
because the tier is what constrains the treatment. This is the confusion matrix to report.

In [ ]:
TIERS = ["negligible", "early", "moderate", "severe", "critical"]
matrix = np.zeros((len(TIERS), len(TIERS)), dtype=int)

for fraction in np.linspace(0.0, 0.8, 40):
    for seed in (1, 2, 3):
        leaf = make_leaf(damaged_fraction=float(fraction),
                         lesion_count=max(int(fraction * 70), 1), seed=seed)
        truth_tier = classify_tier(leaf.true_damaged_fraction * 100)
        report, _ = analyse(leaf.image)
        matrix[TIERS.index(truth_tier.value), TIERS.index(report.action_tier.value)] += 1

total = matrix.sum()
correct = int(np.trace(matrix))
adjacent = int(sum(matrix[i, j] for i in range(5) for j in range(5) if abs(i - j) == 1))

figure, ax = plt.subplots(figsize=(6, 5))
ax.imshow(matrix, cmap="Blues")
ax.set_xticks(range(5), TIERS, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(5), TIERS, fontsize=8)
ax.set_xlabel("measured tier")
ax.set_ylabel("true tier")
for i in range(5):
    for j in range(5):
        if matrix[i, j]:
            ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=9,
                    color="white" if matrix[i, j] > matrix.max() / 2 else "black")
ax.set_title(f"exact {correct}/{total} ({correct / total:.0%}), "
             f"within one tier {(correct + adjacent) / total:.0%}")
figure.tight_layout();

## 4. Stress tests — what breaks colour thresholding

These are the failure modes that separate a lab demo from something usable on a phone in a
field. Each row shows the perturbation, the measured severity, and how far it drifted from
the clean baseline.

In [ ]:
base = make_leaf(damaged_fraction=0.25, lesion_count=15, seed=77)
truth = base.true_damaged_fraction * 100


def warm_cast(img):
    out = img.astype(np.float32) * np.array([0.78, 0.95, 1.22])   # BGR -> orange light
    return np.clip(out, 0, 255).astype(np.uint8)


def cool_cast(img):
    out = img.astype(np.float32) * np.array([1.25, 1.02, 0.8])    # bluish shade
    return np.clip(out, 0, 255).astype(np.uint8)


def underexpose(img):
    return (img.astype(np.float32) * 0.45).astype(np.uint8)


def overexpose(img):
    return np.clip(img.astype(np.float32) * 1.65, 0, 255).astype(np.uint8)


def add_shadow(img):
    out = img.copy()
    h, w = out.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    cv2.ellipse(mask, (int(w * 0.68), int(h * 0.6)), (int(w * 0.3), int(h * 0.42)), 20, 0, 360, 255, -1)
    out[mask > 0] = (out[mask > 0].astype(np.float32) * 0.55).astype(np.uint8)
    return out


def jpeg_artifacts(img):
    ok, buffer = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), 25])
    return cv2.imdecode(buffer, cv2.IMREAD_COLOR) if ok else img


def busy_background(img):
    out = img.copy()
    rng = np.random.default_rng(5)
    soil = rng.integers(40, 110, out.shape, dtype=np.int16)
    soil[:, :, 2] = np.clip(soil[:, :, 2] + 55, 0, 255)    # reddish soil
    outside = base.leaf_mask == 0
    out[outside] = soil[outside].astype(np.uint8)
    return out


def macro_crop(img):
    # Crop inside the blade so the frame is *all* leaf: the case where an adaptive
    # threshold has no background to find and would split the leaf itself.
    ys, xs = np.nonzero(base.leaf_mask)
    return img[ys.min() + 40:ys.max() - 40, xs.min() + 4:xs.max() - 4]


PERTURBATIONS = [
    ("clean", lambda x: x),
    ("warm light", warm_cast),
    ("cool shade", cool_cast),
    ("underexposed", underexpose),
    ("overexposed", overexpose),
    ("hard shadow", add_shadow),
    ("jpeg q25", jpeg_artifacts),
    ("soil background", busy_background),
    ("macro (all leaf)", macro_crop),
]

print(f"ground truth: {truth:.1f} %\n")
print(f"{'perturbation':<20}{'severity':>10}{'drift':>8}{'cover':>8}  tier")
print("-" * 62)

results = []
for name, transform in PERTURBATIONS:
    perturbed = transform(base.image)
    report, masks = analyse(perturbed)
    results.append((name, perturbed, report))
    print(f"{name:<20}{report.severity_pct:9.1f}%{report.severity_pct - truth:+8.1f}"
          f"{report.leaf_coverage_pct:7.1f}%  {report.action_tier.value}")

figure, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, (name, perturbed, report) in zip(axes.ravel(), results, strict=True):
    _, masks = analyse(perturbed)
    show_image(render_overlay(masks), f"{name}\n{report.severity_pct:.1f}% ({report.action_tier.value})", ax=ax)
figure.tight_layout();

### Reading the stress table

This table is why the segmentation looks the way it does. An earlier version of
`segment_leaf` used a single fixed saturation floor, and this exact experiment is what
condemned it:

| Perturbation | Fixed floor | Adaptive + border test |
|---|---|---|
| warm light | **+49.1 pp** (two tiers wrong) | −0.2 pp |
| soil background | **+48.7 pp** (two tiers wrong) | −0.3 pp |
| cool shade | −16.4 pp | −8.9 pp |
| clean / shadow / underexposed / jpeg | within 2 pp | within 2 pp |

The mechanism behind the two catastrophic rows is the same: under a warm cast, or against
soil, the background's saturation climbs past the fixed floor, the leaf mask swallows the
whole frame, and every off-green *background* pixel is counted as diseased tissue. The
severity index does not drift — it inverts, reporting 70 % damage on a 21 % leaf.

The fix generates two candidate masks, the fixed floor and an Otsu threshold that adapts to
the image, and validates each on plausible area plus a **border test**: the region a
candidate excludes must actually lie at the frame edge. That second condition is what makes
Otsu safe. On the `macro (all leaf)` row there is no background to find, so Otsu splits the
leaf itself and carves out the darker lesions — a mask that excludes *interior* pixels while
the border stays leaf. It is rejected, and the whole frame is measured with a quality
warning instead. Note the direction of that failure: it over-reports slightly rather than
silently dropping lesions, which is the right way round for a safety-relevant number.

Remaining known weaknesses, honestly:

| Failure mode | Why it still hurts | Planned answer |
|---|---|---|
| Cool/blue cast | Shifts hue out of the green band before any masking happens; background-referenced white balance only partly recovers it | Classify in LAB rather than HSV, where lightness and chroma separate |
| Overexposure | Blown highlights lose the chroma that distinguishes chlorotic from healthy | Already warns; the app should request a retake |
| Deep shadow inside the leaf | `V < 55` reads dark pixels as necrosis; here the shadow happened to fall mostly outside | Local-contrast normalisation instead of a global value cut |
| Real field clutter | Overlapping leaves defeat largest-connected-component, since the neighbour *is* a leaf | A lightweight segmentation model is the only real answer |

The point of the table is that the severity index has a *stated* operating envelope. A
number with known limits is worth more than a number presented as exact.

## 5. Bring your own annotated leaves

Drop real photos into `data/samples/annotated/` alongside a `truth.csv`:

```
filename,damaged_pct
tomato_field_01.jpg,18.5
tomato_field_02.jpg,42.0
```

The cell below is a no-op until that directory exists, so the notebook still runs clean.
Agreement on real leaves — not synthetic ones — is the number for the evaluation chapter.

In [ ]:
import csv

annotated_dir = ROOT / "data" / "samples" / "annotated"
truth_csv = annotated_dir / "truth.csv"

if not truth_csv.exists():
    print(f"No annotated set yet. Create {truth_csv} to enable this section.")
else:
    truths, measurements, names = [], [], []
    with truth_csv.open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            path = annotated_dir / row["filename"]
            if not path.exists():
                print(f"  missing: {row['filename']}")
                continue
            report, _ = analyse(cv2.imread(str(path)))
            truths.append(float(row["damaged_pct"]))
            measurements.append(report.severity_pct)
            names.append(row["filename"])

    errors = np.abs(np.array(measurements) - np.array(truths))
    print(f"n = {len(truths)}  MAE = {errors.mean():.2f} pp  max = {errors.max():.2f} pp")

    figure, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 100], [0, 100], "--", color="grey", linewidth=1)
    ax.scatter(truths, measurements, s=50, color="#6a1b9a")
    ax.set_xlabel("expert annotation (%)")
    ax.set_ylabel("measured severity (%)")
    ax.set_title(f"Agreement on real leaves (MAE {errors.mean():.2f} pp)")
    ax.grid(alpha=0.2)

## 6. Persisting a tuned configuration

`SeverityConfig` is a frozen dataclass, so a tuned variant is explicit and diffable. Once
the sweeps above justify a change, edit the defaults in `app/vision/severity.py` and record
*why* in the commit — the numbers in that file should always be traceable to a measurement
made here.

In [ ]:
from dataclasses import asdict, replace

tuned = replace(DEFAULT_CONFIG, green_min_saturation=45, dark_value_max=55)

print(f"{'field':<26}{'default':>10}{'tuned':>10}")
print("-" * 46)
for field, value in asdict(DEFAULT_CONFIG).items():
    if field == "tier_bounds":
        continue
    new = asdict(tuned)[field]
    flag = "  <-- changed" if new != value else ""
    print(f"{field:<26}{str(value):>10}{str(new):>10}{flag}")

print(f"\nMAE  default {mean_abs_error(DEFAULT_CONFIG):.2f} pp   tuned {mean_abs_error(tuned):.2f} pp")